# CAKGI — imputing a CGM + heart-rate dataset

This notebook applies **CAKGI** (context-aware, knowledge-guided diffusion imputation; the PIDCK model) to a whole dataset. It uses the two-tier rule from the paper:

| Gap length | Method |
|---|---|
| < 30 min | linear interpolation |
| 30 min – 8 h | CAKGI |
| > 8 h | left missing |

**Pipeline**
1. Load the data and label every missing sample as a small, long or too-long gap. This writes `SDBII_gap_classified.csv`, the input of the Monte Carlo notebook.
2. Build the context vector of every long gap and a canonical gap registry. This writes `SDBII_context_vectors.csv` and `SDBII_canonical_gaps.csv`.
3. Define the CAKGI model.
4. Train CAKGI, or load the checkpoint `SDBII_pidck_v4_model.pt`.
   - 75 epochs; Adam with lr 10⁻³ and batch size 32.
   - Learning rate: 5-epoch warm-up, then cosine decay.
   - Loss: diffusion loss + 0.1 × Cauchy regulariser + 0.3 × boundary loss.
5. Impute the long gaps.
   - Adaptive context window: min(120 + 0.5·duration, 360) min on each side.
   - Interpolation initialisation, adaptive T_eff = clip(2·gap_len, 20, 100), and boundary guidance.
   - The output is the median of 50 chains. When a hypoglycaemic reading occurred less than 30 min before the gap, glucose uses the 25th percentile instead.
   - Each imputed value comes with an 80 % interval (10th–90th percentile of the chains) and the learned position weights λ_pre and λ_post.
6. Linearly interpolate the short gaps (< 30 min).
7. Assemble and verify the output, `SDBII_imputed_pidck.csv`.
   - `GlucoseCGM_imputed_final` and `HR_imputed_final` hold the observed or imputed values.
   - `imputation_method_*` marks each value as `real`, `PIDCK_long`, `linear_short` or `missing`.
   - The CI and λ columns are filled for CAKGI values.

**Input.** `SDBII.csv` with columns `PtID`, `ts`, `GlucoseCGM` (mg/dL) and `HR` (bpm). It must be on a regular 5-min grid, with missing samples as empty or 0.

**Note.** This notebook imputes the full dataset, so its normalisation statistics use all patients. The leakage-free benchmark with held-out patients is `CAKGI_MonteCarlo_Evaluation.ipynb`.

`SMOKE_TEST = True` (Cell 1) runs everything in a few minutes on CPU with tiny settings. It only checks that the code runs; its numbers are meaningless.

**Requirements:** Python ≥ 3.10, PyTorch ≥ 2.0 (CUDA, Apple MPS or CPU), NumPy, pandas ≥ 2.0, SciPy, scikit-learn, Matplotlib, tqdm. Tested with PyTorch 2.14, pandas 2.2 and scikit-learn 1.9.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — Imports and configuration
# ════════════════════════════════════════════════════════════════════
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
warnings.filterwarnings('ignore')

BASE             = r"path/to/your/data_directory"     # <-- set this
ORIGINAL_FILE    = f"{BASE}/SDBII.csv"
GAP_CLASSIFIED   = f"{BASE}/SDBII_gap_classified.csv"    # input of the Monte Carlo notebook
VECTORS_PATH     = f"{BASE}/SDBII_context_vectors.csv"   # input of the Monte Carlo notebook
CANONICAL_PATH   = f"{BASE}/SDBII_canonical_gaps.csv"
MODEL_PATH       = f"{BASE}/SDBII_pidck_v4_model.pt"
CHECKPOINT_PATH  = f"{BASE}/SDBII_imputed_pidck_checkpoint.csv"
FINAL_PATH       = f"{BASE}/SDBII_imputed_pidck.csv"
LONG_DETAIL_PATH = f"{BASE}/SDBII_imputed_pidck_details.csv"

FREQ           = '5min'
CTX_WIN        = 120     # min: context window for the context features
MAX_SAMPLE_WIN = 360     # min: maximum signal window on each side of a gap
MAX_L          = 1500    # max window length (steps) for one diffusion pass
T_STEPS        = 100
N_SAMPLES      = 50      # chains; the output is their median
SEQ_LEN        = 48
MASK_RATIO     = 0.15
EPOCHS         = 75
BATCH_SIZE     = 32
LR             = 1e-3
NF, DM, NH, NL = 2, 64, 4, 3
HYPO_THR       = 70
MIN_GAP_MINS   = 30      # shorter gaps: linear interpolation
MAX_GAP_MINS   = 480     # longer gaps: left missing
CAUCHY_GAMMA   = 0.1
CAUCHY_L       = 10.0
BOUNDARY_W     = 0.3
SEED           = 2026
MAX_TRAIN_WINDOWS = None # None = all windows (smoke test only: cap)
MAX_LONG_GAPS     = None # None = all long gaps (smoke test only: cap)

SMOKE_TEST = False
if SMOKE_TEST:
    EPOCHS, N_SAMPLES, MAX_TRAIN_WINDOWS, MAX_LONG_GAPS = 2, 4, 64, 6

np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"CAKGI (PIDCK v4) | device: {DEVICE} | epochs: {EPOCHS} | chains: {N_SAMPLES} | MAX_L: {MAX_L}"
      + (" | SMOKE TEST" if SMOKE_TEST else ""))

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 2 — Load data and classify every missing sample by the length of its gap
# ════════════════════════════════════════════════════════════════════
df = pd.read_csv(ORIGINAL_FILE, sep=None, engine='python')
df['ts'] = pd.to_datetime(df['ts'], format='mixed', dayfirst=True)
df = df.sort_values(['PtID','ts']).reset_index(drop=True)
df['GlucoseCGM'] = df['GlucoseCGM'].replace(0, np.nan)
df['HR']         = df['HR'].replace(0, np.nan)

G_MEAN = df['GlucoseCGM'].mean(); G_STD = df['GlucoseCGM'].std()
H_MEAN = df['HR'].mean();          H_STD = df['HR'].std()
G_STD  = G_STD if G_STD > 0 else 1.
H_STD  = H_STD if H_STD > 0 else 1.

def classify_gaps(pat_df, col, freq=FREQ, thr=MIN_GAP_MINS):
    """
    Labels EVERY missing timestep in each gap:
      'small'   — gap < 30 min  -> linear interpolation
      'long'    — 30 min <= gap <= MAX_GAP_MINS (8hr) -> PIDCK imputation
      'toolong' — gap > 8hr -> left as missing
    """
    full = pd.date_range(pat_df['ts'].min(), pat_df['ts'].max(), freq=freq)
    s    = pat_df.set_index('ts')[col].reindex(full)
    mis  = s.isna()
    lbl  = pd.Series(None, index=full, dtype=object)
    if mis.any():
        gid = (mis != mis.shift()).cumsum()[mis]
        for g, grp in s[mis].groupby(gid):
            dur_mins = len(grp) * 5
            if dur_mins < thr:
                gap_type = 'small'
            elif dur_mins <= MAX_GAP_MINS:
                gap_type = 'long'
            else:
                gap_type = 'toolong'
            lbl.loc[grp.index] = gap_type
    result = pat_df['ts'].map(lbl[lbl.notna()].to_dict())
    result.index = pat_df.index
    return result

df['GlucoseCGM_gap'] = None; df['HR_gap'] = None
for pid, g in df.groupby('PtID'):
    df.loc[g.index,'GlucoseCGM_gap'] = classify_gaps(g,'GlucoseCGM').values
    df.loc[g.index,'HR_gap']         = classify_gaps(g,'HR').values

df.to_csv(GAP_CLASSIFIED, index=False)
print(f"Loaded {len(df):,} rows | {df['PtID'].nunique()} patients -> {GAP_CLASSIFIED}")
print("Glucose gaps:", df['GlucoseCGM_gap'].value_counts(dropna=True).to_dict())
print("HR gaps:",      df['HR_gap'].value_counts(dropna=True).to_dict())

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 3 — Context vector of every long gap + canonical gap registry
#   Pre-gap: 11 statistics of the 2 h before the gap + duration; the same 11 statistics
#   1, 3 and 7 days earlier (±15 min); 7 contrasts current − history per anchor.
#   Post-gap: 9 statistics of the 2 h after the gap, time of day (sin, cos), duration.
# ════════════════════════════════════════════════════════════════════
HIST_OFFSETS = {'1d': pd.Timedelta(days=1),
                '3d': pd.Timedelta(days=3),
                '7d': pd.Timedelta(days=7)}
HIST_TOL     = pd.Timedelta(minutes=15)


def safe_slope(arr):
    a = arr.dropna() if hasattr(arr, 'dropna') else arr[~np.isnan(arr)]
    return np.polyfit(np.arange(len(a)), a, 1)[0] if len(a) >= 2 else np.nan


def tod_enc(ts):
    h = ts.hour + ts.minute / 60
    return np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)


def extract_win_fast(pat_ts_idx, center_ts, win_mins, prefix):
    """
    Extract 11 clinical features from window [center_ts-win_mins, center_ts).
    pat_ts_idx: patient DataFrame already set_index('ts') and sorted.
    Uses loc[] slicing for O(log n) access instead of boolean filter.
    """
    lo = center_ts - pd.Timedelta(minutes=win_mins)
    hi = center_ts - pd.Timedelta(seconds=1)
    try:
        w = pat_ts_idx.loc[lo:hi]
    except KeyError:
        w = pat_ts_idx.iloc[0:0]

    pg = w['GlucoseCGM']; ph = w['HR']
    sl_g = safe_slope(pg); sl_h = safe_slope(ph)

    phf = int((pg < HYPO_THR).any()) if len(pg) > 0 else 0
    phc = int((pg < HYPO_THR).sum()) if len(pg) > 0 else 0
    phl = np.nan
    if phf:
        ht  = w[w['GlucoseCGM'] < HYPO_THR].index   # ts index
        phl = (center_ts - ht.max()).total_seconds() / 60

    return {
        f'{prefix}_glucose_mean'     : pg.mean() if len(pg) else np.nan,
        f'{prefix}_glucose_min'      : pg.min()  if len(pg) else np.nan,
        f'{prefix}_glucose_last'     : pg.dropna().iloc[-1] if pg.dropna().size > 0 else np.nan,
        f'{prefix}_glucose_slope'    : sl_g,
        f'{prefix}_glucose_dropping' : int(sl_g < -1.) if not np.isnan(sl_g) else 0,
        f'{prefix}_hypo_flag'        : phf,
        f'{prefix}_hypo_count'       : phc,
        f'{prefix}_time_since_hypo'  : phl,
        f'{prefix}_hr_mean'          : ph.mean() if len(ph) else np.nan,
        f'{prefix}_hr_last'          : ph.dropna().iloc[-1] if ph.dropna().size > 0 else np.nan,
        f'{prefix}_hr_slope'         : sl_h,
    }


def extract_hist_contrast_fast(pat_ts_idx, gap_start, current_pre):
    """
    Historical features + contrastive differences for 1d/3d/7d anchors.
    Uses indexed lookup instead of full DataFrame scan per anchor.
    """
    hist_feats = {}; contrast_feats = {}
    core = ['glucose_mean','glucose_min','glucose_last',
            'glucose_slope','hr_mean','hr_last','hr_slope']
    nan_hist_keys = ['glucose_mean','glucose_min','glucose_last','glucose_slope',
                     'glucose_dropping','hypo_flag','hypo_count','time_since_hypo',
                     'hr_mean','hr_last','hr_slope']

    for name, delta in HIST_OFFSETS.items():
        hist_ts = gap_start - delta
        lo = hist_ts - HIST_TOL; hi = hist_ts + HIST_TOL
        try:
            near = pat_ts_idx.loc[lo:hi]
        except KeyError:
            near = pat_ts_idx.iloc[0:0]

        if len(near) == 0:
            for k in nan_hist_keys:
                hist_feats[f'hist_{name}_{k}'] = np.nan
            for k in core:
                contrast_feats[f'contrast_{name}_{k}'] = np.nan
            continue

        # Find closest timestamp to hist_ts
        diffs = np.abs([(t - hist_ts).total_seconds() for t in near.index])
        closest = near.index[int(np.argmin(diffs))]
        hf = extract_win_fast(pat_ts_idx, closest, CTX_WIN, f'hist_{name}')
        hist_feats.update(hf)

        for k in core:
            cur = current_pre.get(f'pre_{k}', np.nan)
            hv  = hf.get(f'hist_{name}_{k}', np.nan)
            try:
                contrast_feats[f'contrast_{name}_{k}'] = float(cur) - float(hv)
            except (TypeError, ValueError):
                contrast_feats[f'contrast_{name}_{k}'] = np.nan

    return hist_feats, contrast_feats


def get_gaps_fast(pat, col, gap_col):
    """
    Get one (gs, ge) tuple per gap by grouping consecutive 'long' runs.
    """
    long_mask = pat[gap_col] == 'long'
    if not long_mask.any():
        return []
    # Group consecutive runs
    gid = (long_mask != long_mask.shift()).cumsum()[long_mask]
    gaps = []
    for _, grp in pat[long_mask].groupby(gid):
        gs = grp['ts'].iloc[0]
        ge = grp['ts'].iloc[-1]
        gaps.append((gs, ge))
    return gaps


def extract_context_vectors(df):
    rows = []
    pat_groups = dict(list(df.groupby('PtID')))

    for pid, pat in tqdm(pat_groups.items(), desc='Context vectors'):
        pat = pat.sort_values('ts').reset_index(drop=True)
        # Build ts-indexed version once per patient for fast slicing
        pat_idx = pat.set_index('ts').sort_index()

        # Collect all unique gap start timestamps
        seen = {}   # gs -> {'ge': ge, 'sig': signal_str}

        for col, gap_col, sig in [('GlucoseCGM','GlucoseCGM_gap','GlucoseCGM'),
                                   ('HR','HR_gap','HR')]:
            for gs, ge in get_gaps_fast(pat, col, gap_col):
                if gs in seen:
                    seen[gs]['ge']  = max(seen[gs]['ge'], ge)
                    seen[gs]['sig'] += f'+{sig}'
                else:
                    seen[gs] = {'ge': ge, 'sig': sig}

        ts_arr = pat['ts'].values   # numpy array for searchsorted

        for gs, info in seen.items():
            ge  = info['ge']
            dur = (ge - gs).total_seconds() / 60 + 5

            pre_feats = extract_win_fast(pat_idx, gs, CTX_WIN, 'pre')
            pre_feats['gap_duration_mins'] = dur

            # Post features — use indexed slice
            ge_ts = pd.Timestamp(ge)
            post_lo = ge_ts + pd.Timedelta(seconds=1)
            post_hi = ge_ts + pd.Timedelta(minutes=CTX_WIN)
            try:
                post = pat_idx.loc[post_lo:post_hi]
            except KeyError:
                post = pat_idx.iloc[0:0]
            og = post['GlucoseCGM']; oh = post['HR']
            sl_pg = safe_slope(og); sl_ph = safe_slope(oh)
            sin_t, cos_t = tod_enc(pd.Timestamp(gs))

            post_feats = {
                'post_glucose_mean'  : og.mean()  if len(og) else np.nan,
                'post_glucose_min'   : og.min()   if len(og) else np.nan,
                'post_glucose_first' : og.dropna().iloc[0] if og.dropna().size > 0 else np.nan,
                'post_glucose_slope' : sl_pg,
                'post_hypo_flag'     : int((og < HYPO_THR).any()) if len(og) else 0,
                'post_hypo_count'    : int((og < HYPO_THR).sum()) if len(og) else 0,
                'post_hr_mean'       : oh.mean()  if len(oh) else np.nan,
                'post_hr_first'      : oh.dropna().iloc[0] if oh.dropna().size > 0 else np.nan,
                'post_hr_slope'      : sl_ph,
                'time_sin'           : sin_t,
                'time_cos'           : cos_t,
                'gap_duration_mins'  : dur,
            }

            hist_feats, contrast_feats = extract_hist_contrast_fast(
                pat_idx, pd.Timestamp(gs), pre_feats)

            row = {'PtID': pid, 'gap_start_ts': gs, 'gap_end_ts': ge,
                   'gap_duration_mins': dur, 'gap_signal': info['sig']}
            row.update(pre_feats)
            row.update(post_feats)
            row.update(hist_feats)
            row.update(contrast_feats)
            rows.append(row)

    vdf = pd.DataFrame(rows)
    vdf['gap_num'] = vdf.groupby('PtID').cumcount() + 1
    print(f"Context vectors: {len(vdf)} gaps | {len(vdf.columns)} columns")
    return vdf


def build_canonical(df, vectors_df):
    # Pre-group df by PtID to avoid repeated full scans
    pat_groups = {pid: grp.sort_values('ts').reset_index(drop=True)
                  for pid, grp in df.groupby('PtID')}
    records = []
    for _, ctx in vectors_df.iterrows():
        pid = ctx['PtID']
        gs  = pd.Timestamp(ctx['gap_start_ts'])
        ge  = pd.Timestamp(ctx['gap_end_ts'])
        dur = ctx['gap_duration_mins']
        pat = pat_groups[pid]
        pat_idx = pat.set_index('ts').sort_index()

        # Gap NaN counts using indexed slice
        try:
            seg = pat_idx.loc[gs:ge]
        except KeyError:
            seg = pat_idx.iloc[0:0]
        ng = int(seg['GlucoseCGM'].isna().sum())
        nh = int(seg['HR'].isna().sum())

        pw    = round(min(CTX_WIN + dur*0.5, MAX_SAMPLE_WIN) / 5) * 5
        pl    = int((pw*2 + dur) / 5) + 2; pskip = pl > MAX_L
        bw    = CTX_WIN
        bl    = int((bw*2 + dur) / 5) + 2; bskip = bl > MAX_L

        try:
            pre  = pat_idx.loc[gs - pd.Timedelta(minutes=bw):gs - pd.Timedelta(seconds=1)]
            post = pat_idx.loc[ge + pd.Timedelta(seconds=1):ge + pd.Timedelta(minutes=bw)]
        except KeyError:
            pre = post = pat_idx.iloc[0:0]

        hpg = bool(pre['GlucoseCGM'].notna().any())
        hog = bool(post['GlucoseCGM'].notna().any())
        hph = bool(pre['HR'].notna().any())
        hoh = bool(post['HR'].notna().any())

        records.append({
            'PtID': pid, 'gap_start_ts': gs, 'gap_end_ts': ge,
            'gap_duration_mins': round(dur, 1), 'gap_signal': ctx['gap_signal'],
            'n_glucose_nan': ng, 'n_hr_nan': nh,
            'pidck_window_mins': round(pw, 1), 'pidck_skips': pskip, 'base_skips': bskip,
            'base_can_impute_g': bool(hpg and hog), 'base_can_impute_h': bool(hph and hoh),
            'pidck_exp_g': 0 if pskip else ng, 'pidck_exp_h': 0 if pskip else nh,
            'base_exp_g':  0 if (bskip or not(hpg and hog)) else ng,
            'base_exp_h':  0 if (bskip or not(hph and hoh)) else nh,
        })

    can = pd.DataFrame(records)
    print(f"Canonical: {len(can)} gaps | PIDCK skips={can['pidck_skips'].sum()}")
    print(f"  PIDCK exp G={can['pidck_exp_g'].sum():,}  H={can['pidck_exp_h'].sum():,}")
    return can


vectors_df = extract_context_vectors(df)
vectors_df.to_csv(VECTORS_PATH, index=False)

canonical = build_canonical(df, vectors_df)
canonical.to_csv(CANONICAL_PATH, index=False)

canonical_with_ctx = canonical.merge(
    vectors_df.drop(columns=['gap_signal','gap_num'], errors='ignore'),
    on=['PtID','gap_start_ts','gap_end_ts','gap_duration_mins'], how='left')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 4 — CAKGI (PIDCK) model
# ════════════════════════════════════════════════════════════════════
# ── Original 12 pre features ─────────────────────────────────────
PRE_COLS_ORIG = [
    'pre_glucose_mean','pre_glucose_min','pre_glucose_last',
    'pre_glucose_slope','pre_glucose_dropping',
    'pre_hypo_flag','pre_hypo_count','pre_time_since_hypo',
    'pre_hr_mean','pre_hr_last','pre_hr_slope','gap_duration_mins'
]

# ── Historical features: 11 per anchor x 3 anchors = 33 ──────────
_HIST_FEATS = ['glucose_mean','glucose_min','glucose_last','glucose_slope',
               'glucose_dropping','hypo_flag','hypo_count','time_since_hypo',
               'hr_mean','hr_last','hr_slope']
PRE_COLS_HIST = [f'hist_{o}_{f}' for o in ['1d','3d','7d'] for f in _HIST_FEATS]

# ── Contrastive features: 7 per anchor x 3 anchors = 21 ──────────
_CONT_FEATS = ['glucose_mean','glucose_min','glucose_last',
               'glucose_slope','hr_mean','hr_last','hr_slope']
PRE_COLS_CONTRAST = [f'contrast_{o}_{f}' for o in ['1d','3d','7d'] for f in _CONT_FEATS]

# ── Full pre/post column lists ────────────────────────────────────
PRE_COLS  = PRE_COLS_ORIG + PRE_COLS_HIST + PRE_COLS_CONTRAST  # 66 total
POST_COLS = ['post_glucose_mean','post_glucose_min','post_glucose_first',
             'post_glucose_slope','post_hypo_flag','post_hypo_count',
             'post_hr_mean','post_hr_first','post_hr_slope','time_sin','time_cos',
             'gap_duration_mins']   # 12 features — gap duration added so post encoder
                                    # knows how strongly to constrain the trajectory

PRE_DIM  = len(PRE_COLS)*2 + 1   # 66*2+1 = 133
POST_DIM = len(POST_COLS)*2       # 12*2   = 24

print(f"PRE_COLS: {len(PRE_COLS)} features -> PRE_DIM={PRE_DIM}")
print(f"POST_COLS: {len(POST_COLS)} features -> POST_DIM={POST_DIM}")


class SinPE(nn.Module):
    def __init__(self,d=DM): super().__init__(); self.d=d
    def forward(self,t):
        h=self.d//2; f=torch.exp(-np.log(10000)*torch.arange(h,device=t.device)/h)
        a=t[:,None].float()*f[None]; return torch.cat([a.sin(),a.cos()],dim=-1)

class CtxEnc(nn.Module):
    def __init__(self,d_in,d=DM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_in,d),nn.LayerNorm(d),nn.GELU(),
                               nn.Linear(d,d),nn.LayerNorm(d),nn.GELU(),nn.Linear(d,d))
    def forward(self,x): return self.net(x).unsqueeze(1)

class PosWeightNet(nn.Module):
    def __init__(self,d=DM):
        super().__init__()
        self.p=nn.Sequential(nn.Linear(d*2+1,d//2),nn.GELU(),nn.Linear(d//2,1))
        self.q=nn.Sequential(nn.Linear(d*2+1,d//2),nn.GELU(),nn.Linear(d//2,1))
    def forward(self,x,cp,cq,gpos):
        B,L,D=x.shape
        fp=torch.cat([x,cp.expand(B,L,D),gpos],dim=-1)
        fq=torch.cat([x,cq.expand(B,L,D),gpos],dim=-1)
        w=F.softmax(torch.cat([self.p(fp)+(1-gpos),self.q(fq)+gpos],dim=-1),dim=-1)
        return w[...,0:1],w[...,1:2]

class DKLayer(nn.Module):
    def __init__(self,d=DM,h=NH):
        super().__init__()
        self.sa=nn.MultiheadAttention(d,h,batch_first=True)
        self.ca1=nn.MultiheadAttention(d,h,batch_first=True)
        self.ca2=nn.MultiheadAttention(d,h,batch_first=True)
        self.pw=PosWeightNet(d)
        self.ff=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Linear(d*4,d))
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.n3=nn.LayerNorm(d)
        self.sa_out=None
    def forward(self,x,cp,cq,gpos):
        x=self.n1(x+self.sa(x,x,x)[0]); self.sa_out=x
        lp,lq=self.pw(x,cp,cq,gpos)
        A1,_=self.ca1(x,cp,cp); A2,_=self.ca2(x,cq,cq)
        x=self.n2(x+lp*A1+lq*A2); x=self.n3(x+self.ff(x)); return x

class PIDCKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.T=T_STEPS
        self.ip=nn.Linear(NF*2,DM); self.te=SinPE(); self.tp=nn.Linear(DM,DM)
        self.pe=CtxEnc(PRE_DIM); self.qe=CtxEnc(POST_DIM)
        self.layers=nn.ModuleList([DKLayer() for _ in range(NL)])
        self.op=nn.Linear(DM,NF)
        b=self._cs(T_STEPS); self.register_buffer('alpha_bar',torch.cumprod(1-b,0))

    def _cs(self,T,s=0.008):
        st=torch.arange(T+1,dtype=torch.float)
        f=torch.cos(((st/T+s)/(1+s))*np.pi/2)**2; f=f/f[0]
        return torch.clamp(1-f[1:]/f[:-1],0.0001,0.9999)

    def _gpos(self,mask,B,L):
        """Relative position inside each missing run (0 at the start, 1 at the end; 0.5 if
        observed or a single step). Device-safe on CUDA, MPS and CPU."""
        obs=mask[:,:,0]; gp=torch.full((B,L),0.5,device=mask.device,dtype=mask.dtype)
        for b in range(B):
            mis=torch.where(obs[b]==0)[0]
            if mis.numel()==0: continue
            brk=(torch.where(torch.diff(mis)>1)[0]+1).cpu().tolist()
            for s,e in zip([0]+brk, brk+[int(mis.numel())]):
                blk=mis[s:e]; n=int(blk.numel())
                gp[b,blk]=torch.linspace(0.,1.,n,device=mask.device,dtype=mask.dtype) if n>1 else 0.5
        return gp.unsqueeze(-1)

    def _cauchy(self,l=CAUCHY_L):
        tot=0.; cnt=0
        for layer in self.layers:
            z=layer.sa_out
            if z is None: continue
            B,L,D=z.shape
            zn=F.normalize(z,dim=-1); sim=torch.bmm(zn,zn.transpose(1,2))
            pos=torch.arange(L,dtype=torch.float,device=z.device)
            tgt=(1./(1.+(pos.unsqueeze(0)-pos.unsqueeze(1))**2/l**2)).unsqueeze(0).expand(B,-1,-1)
            tot+=torch.nn.functional.mse_loss(sim,tgt); cnt+=1
        return tot/max(cnt,1)

    def forward(self,xn,t,mask,cp_v,cq_v):
        B, L, NF_ = xn.shape
        x=self.ip(torch.cat([xn,mask],dim=-1))+self.tp(self.te(t)).unsqueeze(1)
        cp=self.pe(cp_v); cq=self.qe(cq_v); gp=self._gpos(mask,B,L)
        for layer in self.layers: x=layer(x,cp,cq,gp)
        return self.op(x)

    def loss(self,x0,mask,cp_v,cq_v,gamma=CAUCHY_GAMMA):
        # CSDI-style imputation training:
        #   x0    = FULL clean signal (true values at ALL positions,
        #           including the artificially masked targets)
        #   mask  = 1 at conditioning (observed), 0 at target positions
        #   Input = noisy targets + CLEAN observed (matches inference)
        #   Loss  = noise prediction error at TARGET positions only
        B=x0.shape[0]; t=torch.randint(1,self.T+1,(B,),device=x0.device)
        ab=self.alpha_bar[t-1].view(B,1,1); eps=torch.randn_like(x0)
        x_t = ab.sqrt()*x0 + (1-ab).sqrt()*eps      # diffuse FULL clean signal
        x_in = x_t*(1-mask) + x0*mask                # clean observed + noisy targets
        pred=self.forward(x_in,t,mask,cp_v,cq_v)
        tgt=(1-mask).bool()                          # TARGET (gap) positions
        ld=torch.nn.functional.mse_loss(pred[tgt],eps[tgt])
        lc=self._cauchy()
        return ld+gamma*lc, ld.item(), lc.item()


model=PIDCKModel().to(DEVICE)
print(f"PIDCK v4 enhanced — {sum(p.numel() for p in model.parameters()):,} params | {DEVICE}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — Training (or loading the checkpoint)
#   Fully observed 48-step windows (50 % overlap); one contiguous block of 15 % of the steps
#   is masked: 50 % anchored (never touching the window edges), 50 % anywhere. The context of
#   the masked block is computed from the patient's record outside the block.
# ════════════════════════════════════════════════════════════════════
HIST_OFFSETS_TR = {'1d': pd.Timedelta(days=1),
                   '3d': pd.Timedelta(days=3),
                   '7d': pd.Timedelta(days=7)}
HIST_TOL_TR     = pd.Timedelta(minutes=15)


def _safe_slope(s):
    s = s.dropna()
    return np.polyfit(np.arange(len(s)), s, 1)[0] if len(s) >= 2 else np.nan


def _tod_enc(ts):
    h = ts.hour + ts.minute / 60
    return np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)


def _extract_win(pat, center_ts, win_mins, prefix):
    w   = pat[(pat['ts'] >= center_ts - pd.Timedelta(minutes=win_mins)) &
              (pat['ts'] <  center_ts)]
    pg  = w['GlucoseCGM']; ph = w['HR']
    phf = int((pg < HYPO_THR).any()) if len(pg) > 0 else 0
    phc = int((pg < HYPO_THR).sum()) if len(pg) > 0 else 0
    phl = np.nan
    if phf:
        ht  = w[w['GlucoseCGM'] < HYPO_THR]['ts']
        phl = (center_ts - ht.max()).total_seconds() / 60
    return {
        f'{prefix}_glucose_mean'     : pg.mean(),
        f'{prefix}_glucose_min'      : pg.min() if len(pg) > 0 else np.nan,
        f'{prefix}_glucose_last'     : pg.dropna().iloc[-1] if pg.dropna().size > 0 else np.nan,
        f'{prefix}_glucose_slope'    : _safe_slope(pg),
        f'{prefix}_glucose_dropping' : int(_safe_slope(pg) < -1.) if not np.isnan(_safe_slope(pg)) else 0,
        f'{prefix}_hypo_flag'        : phf,
        f'{prefix}_hypo_count'       : phc,
        f'{prefix}_time_since_hypo'  : phl,
        f'{prefix}_hr_mean'          : ph.mean(),
        f'{prefix}_hr_last'          : ph.dropna().iloc[-1] if ph.dropna().size > 0 else np.nan,
        f'{prefix}_hr_slope'         : _safe_slope(ph),
    }


def _hist_contrast(pat, gap_start, current_pre):
    hist_feats = {}; contrast_feats = {}
    core = ['glucose_mean','glucose_min','glucose_last',
            'glucose_slope','hr_mean','hr_last','hr_slope']
    for name, delta in HIST_OFFSETS_TR.items():
        hist_ts = gap_start - delta
        near    = pat[(pat['ts'] >= hist_ts - HIST_TOL_TR) &
                      (pat['ts'] <= hist_ts + HIST_TOL_TR)]
        if len(near) == 0:
            feat_keys = ['glucose_mean','glucose_min','glucose_last','glucose_slope',
                         'glucose_dropping','hypo_flag','hypo_count','time_since_hypo',
                         'hr_mean','hr_last','hr_slope']
            for k in feat_keys:
                hist_feats[f'hist_{name}_{k}'] = np.nan
            for k in core:
                contrast_feats[f'contrast_{name}_{k}'] = np.nan
            continue
        closest = near.iloc[(near['ts'] - hist_ts).abs().argsort()[:1]]['ts'].values[0]
        hf = _extract_win(pat, pd.Timestamp(closest), CTX_WIN, f'hist_{name}')
        hist_feats.update(hf)
        for k in core:
            cur = current_pre.get(f'pre_{k}', np.nan)
            hv  = hf.get(f'hist_{name}_{k}', np.nan)
            try:
                contrast_feats[f'contrast_{name}_{k}'] = float(cur) - float(hv)
            except (TypeError, ValueError):
                contrast_feats[f'contrast_{name}_{k}'] = np.nan
    return hist_feats, contrast_feats


class CGMDataset(Dataset):
    def __init__(self, df, vdf, seq=SEQ_LEN, mr=MASK_RATIO):
        self.seq = seq
        self.mr  = mr
        self.gm, self.gs = df['GlucoseCGM'].mean(), df['GlucoseCGM'].std()
        self.hm, self.hs = df['HR'].mean(),          df['HR'].std()

        pr = vdf[PRE_COLS].copy()
        po = vdf[POST_COLS].copy()
        self.pm, self.ps = pr.mean(), pr.std().replace(0, 1)
        self.qm, self.qs = po.mean(), po.std().replace(0, 1)

        self.patient_data = {
            pid: pat.sort_values('ts').reset_index(drop=True)
            for pid, pat in df.groupby('PtID')
        }

        self.samples = []
        for pid, pat in self.patient_data.items():
            g   = (pat['GlucoseCGM'].values - self.gm) / self.gs
            h   = (pat['HR'].values          - self.hm) / self.hs
            sig = np.stack([g, h], axis=1)
            ts  = pat['ts'].values
            for i in range(0, len(sig) - seq, seq // 2):
                w = sig[i:i+seq]
                if np.isnan(w).any():
                    continue
                self.samples.append((w, ts[i:i+seq], pid))
        if MAX_TRAIN_WINDOWS is not None and len(self.samples) > MAX_TRAIN_WINDOWS:   # smoke test only
            keep = np.random.default_rng(SEED).choice(len(self.samples), MAX_TRAIN_WINDOWS, replace=False)
            self.samples = [self.samples[i] for i in sorted(keep)]
        print(f"Training windows: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def _extract_ctx(self, pid, gap_start_ts, gap_end_ts):
        pat = self.patient_data[pid]
        gs  = pd.Timestamp(gap_start_ts)
        ge  = pd.Timestamp(gap_end_ts)
        dur = (ge - gs).total_seconds() / 60 + 5

        pre_feats = _extract_win(pat, gs, CTX_WIN, 'pre')
        pre_feats['gap_duration_mins'] = dur

        post = pat[(pat['ts'] >  ge) &
                   (pat['ts'] <= ge + pd.Timedelta(minutes=CTX_WIN))]
        og = post['GlucoseCGM']; oh = post['HR']
        sin_t, cos_t = _tod_enc(gs)
        post_vals = np.array([
            og.mean(),
            og.min() if len(og) > 0 else np.nan,
            og.dropna().iloc[0] if og.dropna().size > 0 else np.nan,
            _safe_slope(og),
            int((og < HYPO_THR).any()),
            int((og < HYPO_THR).sum()),
            oh.mean(),
            oh.dropna().iloc[0] if oh.dropna().size > 0 else np.nan,
            _safe_slope(oh),
            sin_t, cos_t,
            dur,   # gap_duration_mins — post encoder learns gap-length-aware constraints
        ], dtype=np.float64)

        hist_feats, contrast_feats = _hist_contrast(pat, gs, pre_feats)

        all_pre = {}
        all_pre.update(pre_feats)
        all_pre.update(hist_feats)
        all_pre.update(contrast_feats)
        pre_vals = np.array([all_pre.get(k, np.nan) for k in PRE_COLS],
                             dtype=np.float64)

        pn      = (pre_vals  - self.pm.values) / self.ps.values
        qn      = (post_vals - self.qm.values) / self.qs.values
        pm_bits = (~np.isnan(pre_vals )).astype(np.float32)
        qm_bits = (~np.isnan(post_vals)).astype(np.float32)
        pf      = np.nan_to_num(pn, nan=0.).astype(np.float32)
        qf      = np.nan_to_num(qn, nan=0.).astype(np.float32)

        return (np.concatenate([pf, pm_bits, [1.]]),
                np.concatenate([qf, qm_bits]))

    def _make_anchored_mask(self, seq):
        """
        Anchored masking: the masked block never touches position 0 or seq-1.
        Ensures both boundary values are always observed.
        """
        n_mask    = max(1, int(seq * self.mr))
        # Start at least 1 from each edge
        max_start = seq - n_mask - 1
        if max_start < 1:
            max_start = 1
        start_idx = int(np.random.randint(1, max_start + 1))
        end_idx   = start_idx + n_mask - 1
        return start_idx, end_idx

    def __getitem__(self, idx):
        w, ts_arr, pid = self.samples[idx]
        x = torch.tensor(w, dtype=torch.float32)

        # 50% anchored masking, 50% free masking
        if np.random.random() < 0.5:
            start_idx, end_idx = self._make_anchored_mask(self.seq)
        else:
            n_mask    = max(1, int(self.seq * self.mr))
            max_start = self.seq - n_mask
            start_idx = int(np.random.randint(0, max_start + 1))
            end_idx   = start_idx + n_mask - 1

        m = torch.ones_like(x)
        m[start_idx:end_idx + 1] = 0.0

        pc, qc = self._extract_ctx(pid, ts_arr[start_idx], ts_arr[end_idx])

        # Boundary position indices (within 3 steps of gap edges)
        bnd = torch.zeros(self.seq, dtype=torch.float32)
        for bi in range(start_idx, end_idx + 1):
            dist = min(bi - start_idx, end_idx - bi)
            if dist < 3:
                bnd[bi] = 1.0 - dist / 3.0   # weight: 1 at edge, 0.33 at dist=2

        return (x, m,
                torch.tensor(pc, dtype=torch.float32),
                torch.tensor(qc, dtype=torch.float32),
                bnd)



dataset = CGMDataset(df, vectors_df)

# ── Load checkpoint if exists (skip training) ────────────────────
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print(f"Model loaded: {MODEL_PATH} imputation ready.")
else:
    print("No saved model found — training from scratch...")
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    opt        = torch.optim.Adam(model.parameters(), lr=LR)

    def lr_lambda(ep):
        if ep < 5: return (ep + 1) / 5
        return 1.0

    warmup_sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    cosine_sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)   # stepped after the warm-up

    model.train()
    for ep in range(EPOCHS):
        tot = dif = cau = bnd_loss_tot = 0.
        for batch in loader:
            x, m, pc, qc, bnd = batch
            x, m, pc, qc, bnd = (x.to(DEVICE), m.to(DEVICE),
                                  pc.to(DEVICE), qc.to(DEVICE), bnd.to(DEVICE))
            opt.zero_grad()
            loss, ld, lc = model.loss(x, m, pc, qc)
            B = x.shape[0]
            t_b   = torch.randint(1, model.T + 1, (B,), device=DEVICE)
            ab_b  = model.alpha_bar[t_b - 1].view(B, 1, 1)
            eps_b = torch.randn_like(x)
            xt_b  = ab_b.sqrt() * x + (1 - ab_b).sqrt() * eps_b
            xin_b = xt_b*(1-m) + x*m
            pred_b = model(xin_b, t_b, m, pc, qc)
            bnd_weight = bnd.unsqueeze(-1).expand_as(eps_b)
            bnd_err    = ((pred_b - eps_b)**2 * bnd_weight * (1-m))
            bnd_loss   = bnd_err.sum() / (bnd_weight*(1-m)).sum().clamp(min=1)
            total = loss + BOUNDARY_W * bnd_loss
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += total.item(); dif += ld; cau += lc
            bnd_loss_tot += bnd_loss.item()

        if ep < 5:
            warmup_sch.step()
        else:
            cosine_sch.step()

        if ep == 0 or (ep + 1) % 10 == 0 or (ep + 1) == EPOCHS:
            n = len(loader)
            warn = (" <- Cauchy>>Diffusion, reduce CAUCHY_GAMMA"
                    if cau/n > dif/n else "")
            print(f"Ep {ep+1:>3}/{EPOCHS} | Total {tot/n:.4f} "
                  f"| Diff {dif/n:.4f} | Cauchy {cau/n:.4f} "
                  f"| Bnd {bnd_loss_tot/n:.4f}{warn}")

    torch.save(model.state_dict(), MODEL_PATH)
    model.eval()
    print(f"Model saved: {MODEL_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 — Long gaps (30 min – 8 h): CAKGI imputation
# ════════════════════════════════════════════════════════════════════

def adaptive_win(dur):
    win = min(CTX_WIN + dur * 0.5, MAX_SAMPLE_WIN)
    # Safety: ensure total window (win + dur + win) stays under MAX_L timesteps
    total_steps = (win * 2 + dur) / 5
    if total_steps > MAX_L - 50:   # 50-step safety margin
        win = max(CTX_WIN, (MAX_L - 50 - dur/5) * 5 / 2)
    return max(win, CTX_WIN)   # never less than 120 min


def adaptive_T(gap_len_steps):
    """Fewer steps for short gaps — boundary constraints dominate."""
    return int(np.clip(gap_len_steps * 2, 20, T_STEPS))


def linear_interp_prior(sig, mask):
    """Linear interpolation between boundary values for gap positions."""
    out = sig.copy(); L, F = sig.shape
    for f in range(F):
        arr = sig[:, f].copy()
        obs = np.where(mask[:, f] > 0)[0]
        if len(obs) < 2: continue
        gap_pos = np.where(mask[:, f] == 0)[0]
        if len(gap_pos) == 0: continue
        pre_obs  = obs[obs < gap_pos[0]]
        post_obs = obs[obs > gap_pos[-1]]
        if len(pre_obs) == 0 or len(post_obs) == 0: continue
        v0=arr[pre_obs[-1]]; i0=pre_obs[-1]
        v1=arr[post_obs[0]]; i1=post_obs[0]
        for i in gap_pos:
            out[i, f] = v0 + (v1 - v0) * (i - i0) / (i1 - i0)
    return out


def build_ctx_vec(ctx_row):
    pv = np.array([ctx_row.get(c, np.nan) if hasattr(ctx_row, 'get')
                   else getattr(ctx_row, c, np.nan)
                   for c in PRE_COLS], dtype=np.float64)
    qv = np.array([ctx_row.get(c, np.nan) if hasattr(ctx_row, 'get')
                   else getattr(ctx_row, c, np.nan)
                   for c in POST_COLS], dtype=np.float64)
    pn  = (pv - dataset.pm.values) / dataset.ps.values
    qn  = (qv - dataset.qm.values) / dataset.qs.values
    pm  = (~np.isnan(pv)).astype(np.float32)
    qm  = (~np.isnan(qv)).astype(np.float32)
    pf  = np.nan_to_num(pn, nan=0.).astype(np.float32)
    qf  = np.nan_to_num(qn, nan=0.).astype(np.float32)
    return np.concatenate([pf, pm, [1.]]), np.concatenate([qf, qm])


def collapse(samps_gap, x_init_np, mask_np, ctx_row):
    """
    Median-of-chains aggregation over the N_SAMPLES chains (CI bands: 10th/90th percentile,
    computed in impute_long_gaps).
    samps_gap : (N_chains, gap_len, NF)
    Returns: (gap_len, NF) median prediction.
    """
    # Median across chains (primary output)
    col = np.median(samps_gap, axis=0)   # (GL, NF)

    # Hypo-aware: recent hypo within 30 min -> conservative p25 for glucose
    since = (ctx_row.get('pre_time_since_hypo', np.nan)
             if hasattr(ctx_row, 'get')
             else getattr(ctx_row, 'pre_time_since_hypo', np.nan))
    if not np.isnan(since) and since < 30:
        col[:, 0] = np.percentile(samps_gap[:, :, 0], 25, axis=0)

    return col


def sample_ddpm(model, x_init, mask, cp, cq, prior, gap_idx,
                n=N_SAMPLES, T_eff=None):
    """
    DDPM reverse sampling with:
    - Informed initialisation from linear prior
    - Adaptive T based on gap length
    - Boundary soft constraint in early reverse steps
    """
    model.eval(); B, L, NF_ = x_init.shape
    if L > MAX_L: return None
    if T_eff is None: T_eff = model.T

    # Boundary weight mask (positions within 2 steps of gap edge)
    bnd_mask = torch.zeros(B, L, NF_, device=DEVICE)  # NF_ avoids shadowing F
    if len(gap_idx) > 0:
        for bi in range(min(2, len(gap_idx))):
            bnd_mask[:, gap_idx[bi],   :] = 0.8 - bi * 0.3
            bnd_mask[:, gap_idx[-(bi+1)], :] = 0.8 - bi * 0.3

    samps = []
    with torch.no_grad():
        for _ in range(n):
            # Informed init: linear prior + noise scaled to T_eff
            x = prior.clone()
            noise_scale = model.alpha_bar[T_eff - 1].sqrt()
            x = x + (1 - noise_scale) * torch.randn_like(x)
            x = x * (1 - mask) + x_init * mask

            for t in reversed(range(1, T_eff + 1)):
                tt  = torch.full((B,), t, device=DEVICE, dtype=torch.long)
                eps = model(x, tt, mask, cp, cq)
                ab  = model.alpha_bar[t - 1]
                abp = (model.alpha_bar[t - 2] if t > 1
                       else torch.tensor(1., device=DEVICE))
                x0  = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-3, 3)
                x   = (abp.sqrt() * x0 + (1 - abp).sqrt() * torch.randn_like(x)
                       if t > 1 else x0)

                # Soft boundary pull in early (high-noise) steps
                if t > T_eff // 2:
                    pull = (t / T_eff) * 0.15
                    x    = x - bnd_mask * pull * (x - prior)

                x = x * (1 - mask) + x_init * mask

            samps.append(x.cpu()); del x, x0, eps
    gc.collect()
    return torch.stack(samps, dim=0)


def impute_long_gaps(df, ctx_df, model, dataset, start_from=0):
    results = []; skipped = []
    if start_from > 0 and os.path.exists(CHECKPOINT_PATH):
        results = pd.read_csv(CHECKPOINT_PATH).to_dict('records')
        print(f"Resumed: {len(results)} timesteps")

    for i, ctx in tqdm(ctx_df.iloc[start_from:].iterrows(),
                        total=len(ctx_df) - start_from, desc='PIDCK v4'):
        pid = ctx['PtID']                         # read pid/gs BEFORE any continue
        gs  = pd.Timestamp(ctx['gap_start_ts'])
        if ctx.get('pidck_skips', False):
            skipped.append({'PtID': pid, 'gap_start_ts': gs,
                           'reason': 'pidck_skips=True (window too large)'})
            continue
        ge  = pd.Timestamp(ctx['gap_end_ts'])
        dur = ctx['gap_duration_mins']
        pat = df[df['PtID'] == pid].sort_values('ts')

        wm   = adaptive_win(dur)
        wm   = round(wm / 5) * 5   # MUST be multiple of 5 — otherwise fidx
                                     # misses original timestamps -> merge fails
        ws   = gs - pd.Timedelta(minutes=wm)
        we   = ge + pd.Timedelta(minutes=wm)
        fidx = pd.date_range(ws, we, freq=FREQ); L = len(fidx)
        if L > MAX_L:
            skipped.append({'PtID': pid, 'gap_start_ts': gs, 'reason': f'L={L}'})
            continue

        seg   = pat[(pat['ts'] >= ws) & (pat['ts'] <= we)]
        g_raw = seg.set_index('ts')['GlucoseCGM'].reindex(fidx)
        h_raw = seg.set_index('ts')['HR'].reindex(fidx)
        g_n   = (g_raw.values - dataset.gm) / dataset.gs
        h_n   = (h_raw.values - dataset.hm) / dataset.hs
        sig   = np.stack([g_n, h_n], axis=1)
        mnp   = (~np.isnan(sig)).astype(np.float32)
        sc    = np.nan_to_num(sig, nan=0.)

        prior_np = linear_interp_prior(sc, mnp)
        xi    = torch.tensor(sc,       dtype=torch.float32).unsqueeze(0).to(DEVICE)
        mt    = torch.tensor(mnp,      dtype=torch.float32).unsqueeze(0).to(DEVICE)
        prior = torch.tensor(prior_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        pv, qv = build_ctx_vec(ctx)
        cp = torch.tensor(pv, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        cq = torch.tensor(qv, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        gidx  = np.where((fidx >= gs) & (fidx <= ge))[0]
        T_eff = adaptive_T(len(gidx))

        try:
            samps = sample_ddpm(model, xi, mt, cp, cq, prior, gidx, T_eff=T_eff)
        except RuntimeError as e:
            skipped.append({'PtID': pid, 'gap_start_ts': gs, 'reason': str(e)})
            gc.collect(); continue
        if samps is None:
            skipped.append({'PtID': pid, 'gap_start_ts': gs, 'reason': 'L>MAX_L'})
            continue

        sn  = samps[:, 0, :, :].numpy()   # (N_chains, L, NF)
        col = collapse(sn[:, gidx, :], sc, mnp, ctx)
        gi  = col[:, 0] * dataset.gs + dataset.gm
        hi  = col[:, 1] * dataset.hs + dataset.hm
        # CI from all chains at gap positions
        glo = np.percentile(sn[:, gidx, 0], 10, axis=0) * dataset.gs + dataset.gm
        ghi = np.percentile(sn[:, gidx, 0], 90, axis=0) * dataset.gs + dataset.gm
        hlo = np.percentile(sn[:, gidx, 1], 10, axis=0) * dataset.hs + dataset.hm
        hhi = np.percentile(sn[:, gidx, 1], 90, axis=0) * dataset.hs + dataset.hm

        with torch.no_grad():
            xp  = model.ip(torch.cat([xi, mt], dim=-1))
            cpe = model.pe(cp); cqe = model.qe(cq)
            gp  = model._gpos(mt, 1, L)
            lp, lq = model.layers[0].pw(xp, cpe, cqe, gp)
            lp_g = lp[0, gidx, 0].cpu().numpy()
            lq_g = lq[0, gidx, 0].cpu().numpy()

        for j, ts in enumerate(fidx[gidx]):
            wi = gidx[j]
            if mnp[wi, 0] == 0:
                results.append({
                    'PtID': pid, 'ts': ts, 'column': 'GlucoseCGM',
                    'GlucoseCGM_imputed': round(float(gi[j]), 3),
                    'glucose_ci_lo': round(float(glo[j]), 3),
                    'glucose_ci_hi': round(float(ghi[j]), 3),
                    'gap_duration_mins': round(float(dur), 1),
                    'window_mins_used': round(float(wm), 1),
                    'T_eff': T_eff, 'n_chains': N_SAMPLES,
                    'gap_signal': ctx.get('gap_signal', 'GlucoseCGM+HR'),
                    'imputation_method': 'PIDCK_long',
                    'lambda_pre': round(float(lp_g[j]), 4),
                    'lambda_post': round(float(lq_g[j]), 4)})
            if mnp[wi, 1] == 0:
                results.append({
                    'PtID': pid, 'ts': ts, 'column': 'HR',
                    'HR_imputed': round(float(hi[j]), 3),
                    'hr_ci_lo': round(float(hlo[j]), 3),
                    'hr_ci_hi': round(float(hhi[j]), 3),
                    'gap_duration_mins': round(float(dur), 1),
                    'window_mins_used': round(float(wm), 1),
                    'T_eff': T_eff, 'n_chains': N_SAMPLES,
                    'gap_signal': ctx.get('gap_signal', 'GlucoseCGM+HR'),
                    'imputation_method': 'PIDCK_long',
                    'lambda_pre': round(float(lp_g[j]), 4),
                    'lambda_post': round(float(lq_g[j]), 4)})

        if (i + 1) % 50 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
            print(f"  Checkpoint {i+1}")

        del xi, mt, cp, cq, prior, samps, sn, xp, cpe, cqe, gp, lp, lq
        gc.collect()

    rdf = pd.DataFrame(results)
    rdf.to_csv(CHECKPOINT_PATH, index=False)
    if skipped:
        print(f"Skipped {len(skipped)} gaps")
    print(f"Imputed: {len(rdf):,} rows")
    return rdf


_gaps_to_impute = canonical_with_ctx if MAX_LONG_GAPS is None else canonical_with_ctx.iloc[:MAX_LONG_GAPS]
long_results_df = impute_long_gaps(df, _gaps_to_impute, model, dataset, start_from=0)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 7 — Short gaps (< 30 min): linear interpolation
# ════════════════════════════════════════════════════════════════════
def impute_short_gaps(df, freq=FREQ):
    """
    Linear interpolation for gaps < 30 min (< 6 timesteps = 'small').
    Modifies GlucoseCGM and HR columns directly.
    Also records which (PtID, ts, col) positions were short-gap filled
    """
    df_out = df.copy()
    short_records = []   # track filled positions for label assignment

    for pid, pat in df_out.groupby('PtID'):
        pat_sorted = pat.sort_values('ts')
        for col, gap_col in [('GlucoseCGM','GlucoseCGM_gap'),('HR','HR_gap')]:
            # Find positions labelled 'small'
            small_mask = pat_sorted[gap_col] == 'small'
            if not small_mask.any(): continue

            # Interpolate the full series then keep only 'small' gap positions
            s  = pat_sorted.set_index('ts')[col]
            si = s.interpolate(method='linear', limit_area='inside')

            # Only fill where: (a) originally NaN, (b) gap_col == 'small'
            for idx in pat_sorted[small_mask].index:
                ts_val = pat_sorted.loc[idx, 'ts']
                if pd.isna(pat_sorted.loc[idx, col]):
                    filled_val = si.get(ts_val, np.nan)
                    if not pd.isna(filled_val):
                        df_out.loc[idx, col] = filled_val
                        short_records.append({
                            'PtID': pid, 'ts': ts_val, 'col': col,
                            'value': round(float(filled_val), 3)
                        })

    short_df = pd.DataFrame(short_records) if short_records else pd.DataFrame(
        columns=['PtID','ts','col','value'])
    n_g = (short_df['col']=='GlucoseCGM').sum() if len(short_df) else 0
    n_h = (short_df['col']=='HR').sum() if len(short_df) else 0
    print(f"Short gaps filled: Glucose={n_g:,} | HR={n_h:,} timesteps")
    return df_out.reset_index(drop=True), short_df

df_short, short_df = impute_short_gaps(df)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 8 — Final assembly and verification
# ════════════════════════════════════════════════════════════════════
df_final = df.copy()   

df_final['GlucoseCGM_imputed_final'] = df_final['GlucoseCGM']
df_final['HR_imputed_final']         = df_final['HR']
df_final['imputation_method_glucose'] = np.where(df_final['GlucoseCGM'].isna(), 'missing', 'real')
df_final['imputation_method_hr']      = np.where(df_final['HR'].isna(),         'missing', 'real')

if len(long_results_df) > 0:
    # ── Fix ts dtype mismatch ─────────────────────────────────────
    long_results_df['ts'] = pd.to_datetime(long_results_df['ts'])
    df_final['ts']        = pd.to_datetime(df_final['ts'])

    # ── Glucose rows ──────────────────────────────────────────────
    lr_g = long_results_df[long_results_df['column'] == 'GlucoseCGM'].copy()
    lr_g = lr_g.drop_duplicates(subset=['PtID', 'ts'], keep='last')
    gcols = [c for c in ['PtID','ts','GlucoseCGM_imputed','glucose_ci_lo','glucose_ci_hi',
                          'gap_duration_mins','window_mins_used','gap_signal',
                          'lambda_pre','lambda_post'] if c in lr_g.columns]
    df_final = df_final.merge(lr_g[gcols], on=['PtID','ts'], how='left')
    gm = df_final['GlucoseCGM_imputed'].notna() & df_final['GlucoseCGM_imputed_final'].isna()
    df_final.loc[gm, 'GlucoseCGM_imputed_final']  = df_final.loc[gm, 'GlucoseCGM_imputed']
    df_final.loc[gm, 'imputation_method_glucose'] = 'PIDCK_long'
    df_final.drop(columns=['GlucoseCGM_imputed'], inplace=True, errors='ignore')

    # ── HR rows ───────────────────────────────────────────────────
    lr_h = long_results_df[long_results_df['column'] == 'HR'].copy()
    lr_h = lr_h.drop_duplicates(subset=['PtID', 'ts'], keep='last')
    hcols = [c for c in ['PtID','ts','HR_imputed','hr_ci_lo','hr_ci_hi'] if c in lr_h.columns]
    df_final = df_final.merge(lr_h[hcols], on=['PtID','ts'], how='left')
    hm = df_final['HR_imputed'].notna() & df_final['HR_imputed_final'].isna()
    df_final.loc[hm, 'HR_imputed_final']    = df_final.loc[hm, 'HR_imputed']
    df_final.loc[hm, 'imputation_method_hr'] = 'PIDCK_long'
    df_final.drop(columns=['HR_imputed'], inplace=True, errors='ignore')

if not short_df.empty:
    sg = short_df[short_df['col'] == 'GlucoseCGM'][['PtID','ts','value']]\
         .rename(columns={'value':'_gs'})
    sh = short_df[short_df['col'] == 'HR'][['PtID','ts','value']]\
         .rename(columns={'value':'_hs'})
    sg['ts'] = pd.to_datetime(sg['ts'])
    sh['ts'] = pd.to_datetime(sh['ts'])
    df_final = df_final.merge(sg, on=['PtID','ts'], how='left')
    df_final = df_final.merge(sh, on=['PtID','ts'], how='left')
    gms = df_final['_gs'].notna() & df_final['GlucoseCGM_imputed_final'].isna()
    hms = df_final['_hs'].notna() & df_final['HR_imputed_final'].isna()
    df_final.loc[gms, 'GlucoseCGM_imputed_final']  = df_final.loc[gms, '_gs']
    df_final.loc[gms, 'imputation_method_glucose'] = 'linear_short'
    df_final.loc[hms, 'HR_imputed_final']           = df_final.loc[hms, '_hs']
    df_final.loc[hms, 'imputation_method_hr']       = 'linear_short'
    df_final.drop(columns=['_gs','_hs'], inplace=True)

for c in ['glucose_ci_lo','glucose_ci_hi','hr_ci_lo','hr_ci_hi',
          'lambda_pre','lambda_post','gap_duration_mins','window_mins_used']:
    if c not in df_final.columns:
        df_final[c] = np.nan

df_final.to_csv(FINAL_PATH, index=False)
long_results_df.to_csv(LONG_DETAIL_PATH, index=False)

print(f"\n{'='*60}")
print(f"Saved: {FINAL_PATH}")
print(f"\nGlucose methods:\n{df_final['imputation_method_glucose'].value_counts().to_string()}")
print(f"\nHR methods:\n{df_final['imputation_method_hr'].value_counts().to_string()}")
print(f"\nNaN remaining: G={df_final['GlucoseCGM_imputed_final'].isna().sum():,}  "
      f"HR={df_final['HR_imputed_final'].isna().sum():,}")

# Expected = unique missing (PtID, ts) positions inside the CAKGI gaps that were processed.
# (A glucose gap and an HR gap that overlap share positions; the sum of the per-gap counts
#  pidck_exp_g / pidck_exp_h in the canonical registry counts such positions twice.)
can_chk = pd.read_csv(CANONICAL_PATH, parse_dates=['gap_start_ts', 'gap_end_ts'])
if MAX_LONG_GAPS is not None:
    can_chk = can_chk.iloc[:MAX_LONG_GAPS]
can_chk = can_chk[~can_chk['pidck_skips'].astype(bool)]
_pos = {'GlucoseCGM': set(), 'HR': set()}
_pats = {pid: p.set_index('ts').sort_index() for pid, p in df.groupby('PtID')}
for _, g in can_chk.iterrows():
    seg = _pats[g['PtID']].loc[g['gap_start_ts']:g['gap_end_ts']]
    for col in _pos:
        _pos[col].update((g['PtID'], t) for t in seg.index[seg[col].isna()])
eg, eh = len(_pos['GlucoseCGM']), len(_pos['HR'])
ag = (df_final['imputation_method_glucose'] == 'PIDCK_long').sum()
ah = (df_final['imputation_method_hr']      == 'PIDCK_long').sum()
print(f"\n[VERIFICATION] CAKGI-imputed vs missing positions inside the processed long gaps")
print(f"  Glucose: actual={ag:,} expected={eg:,}  {'OK' if abs(ag-eg)/max(eg,1)<0.02 else 'WARN'}")
print(f"  HR:      actual={ah:,} expected={eh:,}  {'OK' if abs(ah-eh)/max(eh,1)<0.02 else 'WARN'}")
print(f"  Canonical registry (per-gap counts, overlaps counted twice): "
      f"G={int(pd.read_csv(CANONICAL_PATH)['pidck_exp_g'].sum()):,} H={int(pd.read_csv(CANONICAL_PATH)['pidck_exp_h'].sum()):,}")
print(f"\nDone. Canonical registry at: {CANONICAL_PATH}")